In [6]:
# ============================================================
# SECTION 0 — Imports and reproducibility
# ============================================================
import os
import random
import warnings
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import networkx as nx
import igraph as ig
import community as community_louvain
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

In [7]:
# ============================================================
# SECTION 1 — Load polblogs.gml
# ============================================================
GML_PATH = "polblogs.gml"

def load_graph(path=GML_PATH):
    """
    Load polblogs.gml.
    The raw file contains duplicate edges, so NetworkX's default simple-graph
    reader raises. We route through igraph, which silently collapses duplicates,
    and rebuild a NetworkX DiGraph with all node attributes preserved.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing {path}. Download polblogs.zip from "
            "https://public.websites.umich.edu/~mejn/netdata/ and unzip here."
        )
    try:
        return nx.read_gml(path, label="id")
    except nx.NetworkXError:
        # Duplicate edges: route through igraph
        g_ig = ig.Graph.Read_GML(path)
        g = nx.DiGraph()
        for v in g_ig.vs:
            attrs = {k: v[k] for k in v.attributes() if v[k] is not None}
            node_id = int(attrs.get("id", v.index))
            g.add_node(node_id, **attrs)
        for e in g_ig.es:
            s = int(g_ig.vs[e.source]["id"])
            t = int(g_ig.vs[e.target]["id"])
            g.add_edge(s, t)
        return g

G_raw = load_graph()

print("=== RAW GRAPH ===")
print(f"Nodes    : {G_raw.number_of_nodes()}")
print(f"Edges    : {G_raw.number_of_edges()}")
print(f"Directed : {G_raw.is_directed()}")
print(f"Sample node attrs: {list(G_raw.nodes(data=True))[0]}")

=== RAW GRAPH ===
Nodes    : 1490
Edges    : 19025
Directed : True
Sample node attrs: (1, {'id': 1.0, 'label': '100monkeystyping.com', 'value': 0.0, 'source': 'Blogarama'})
